## JSON Parsing and Processing

In [1]:
import json
import os

## Json Processing Stratergies

In [2]:
## Importing the JSON files
from langchain_community.document_loaders import JSONLoader
import json

f:\D Drive\Udemy_RAG (Krish Naik)\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
## Method-I : JsonLoader with jq_schema
print(" JSON Loader - Extract Specific Fields")
employee_loader = JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema='.employees[]', # jq query is mainly used to extract each employee in JSON Data
    text_content=False ## Get full JSON Objects
)
employee_docs = employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First Employee: {employee_docs[0].page_content[:200]} ...")
print(employee_docs)

 JSON Loader - Extract Specific Fields
Loaded 2 employee documents
First Employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status" ...
[Document(metadata={'source': 'F:\\D Drive\\Udemy_RAG (Krish Naik)\\0-DataIngestonParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'F:\\D Drive\\Udemy_RAG (Krish Naik)\\0-DataIngestonParsing\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "

In [7]:
## Method-II: Custom JSON Processing for Complex Structures
from typing import List
from langchain_core.documents import Document
print("Custom JSON Processing")
def process_json_intelligently(filePath: str) -> List[Document]:
    """ Processing the JSON Data with Intelligent Flattening and Context Preservation"""
    with open(filePath, 'r') as f:
        data = json.load(f)

    documents = []
    for emp in data.get('employees', []):
        content = f"""
            Employee Profile:
                Name: {emp['name']}
                Role: {emp['role']}
                Skills: {','.join(emp['skills'])}
                Projects: """
        for proj in emp.get('projects',[]):
            content = content + f"\n- {proj['name']} (Status: {proj['status']})"
        doc = Document(
            page_content=content,
            metadata = {
                'source':filePath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)
    return documents


Custom JSON Processing


In [8]:
process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='\n            Employee Profile:\n                Name: John Doe\n                Role: Software Engineer\n                Skills: Python,JavaScript,React\n                Projects: \n- RAG System (Status: In Progress)\n- Data Pipeline (Status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='\n            Employee Profile:\n                Name: Jane Smith\n                Role: Data Scientist\n                Skills: Python,Machine Learning,SQL\n                Projects: \n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)')]

In [15]:
from typing import List
from langchain_core.documents import Document
import json

def processing_jsonl_documents(filePath: str) -> List[Document]:
    documents = []
    with open(filePath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            info = json.loads(line)
            content = f"""
Event-Details:
    Timestamp: {info.get('timestamp', '')}
    Event: {info.get('event', '')}
    User ID: {info.get('user_id', '')}
"""
            if info.get('page'):
                content += f"Page Details: {info['page']}\n"
            if info.get('amount'):
                content += f"Amount Details: {info['amount']}\n"
            doc = Document(
                page_content=content,
                metadata={
                    'source': filePath,
                    'data_type': 'event_details',
                    'timestamp': info.get('timestamp', ''),
                    'event': info.get('event', ''),
                    'user_id': info.get('user_id', '')
                }
            )
            documents.append(doc)
    return documents

In [16]:
processing_json_documents("data/json_files/events.jsonl")

JSONDecodeError: Extra data: line 2 column 1 (char 67)